# symspellpy とマスク言語モデルで英語のスペル誤りを文脈に合わせて直す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_gec_education/spelling_correction.ipynb)

NLP クックブック 27-15 ／ 文法誤り訂正と言語教育

英文のつづりの誤りを、辞書にない語 (recieved) だけでなく、実在する別の語になってしまった誤り (a peace of cake) も含めて直したい。速い辞書引きで候補を出し、文脈に合う語をマスク言語モデルで選ぶ（1 語ずつ手軽に直すだけなら、この章の「英語のスペルミスをライブラリで直す (pyspellchecker)」）。

In [1]:
!pip install -q symspellpy==6.10.2 pyspellchecker==0.9.1
# 評価用: Peter Norvig が Birkbeck spelling error corpus (Roger Mitton) から取り出した (誤り, 正解) の組。ライセンスの記載はない
!wget -nc -q https://norvig.com/spell-testset1.txt

In [2]:
import importlib.resources
from symspellpy import SymSpell, Verbosity

# 同梱の英語の辞書 (約 8 万語と頻度)。SymSpell の README によると、Google Books Ngram (CC BY 3.0) と SCOWL の両方にある語から作られている
sym = SymSpell(max_dictionary_edit_distance=2)
res = importlib.resources.files("symspellpy")
sym.load_dictionary(str(res / "frequency_dictionary_en_82_765.txt"), term_index=0, count_index=1)
sym.load_bigram_dictionary(str(res / "frequency_bigramdictionary_en_243_342.txt"), term_index=0, count_index=2)

# 1 語ずつ: 編集距離が最も小さい候補を、頻度の高い順に返す
for w in ["speling", "recieve", "definately", "inconvient", "teh", "peace"]:
    print(w, "→", [(h.term, h.distance, h.count) for h in sym.lookup(w, Verbosity.CLOSEST, max_edit_distance=2)[:4]])

# 文全体: 語の分割・結合の誤り (whereis, th elove) もまとめて直す
for s in ["whereis th elove hehad dated forImuch of thepast", "I recieved a wonderfull mesage from my freind"]:
    print(s, "\n →", sym.lookup_compound(s, max_edit_distance=2)[0].term)

# 空白のない文字列を単語に分ける
print(sym.word_segmentation("thequickbrownfoxjumpsoverthelazydog").corrected_string)

speling → [('spelling', 1, 7368045), ('spewing', 1, 273406)]
recieve → [('receive', 1, 88328938), ('relieve', 1, 3018810)]
definately → [('definitely', 1, 15922257)]
inconvient → [('inconvenient', 2, 681396), ('incontinent', 2, 120488)]
teh → [('the', 1, 23135851162), ('tech', 1, 93401669), ('tel', 1, 60827708), ('ten', 1, 46907473)]
peace → [('peace', 0, 47856201)]
whereis th elove hehad dated forImuch of thepast 
 → where is the love he had dated for much of the past
I recieved a wonderfull mesage from my freind 
 → i received a wonderful message from my friend
the quick brown fox jumps over the lazy dog


In [3]:
# 1 語ずつの訂正の正解率と速さを pyspellchecker と比べる
import time
from spellchecker import SpellChecker

tests = [(wrong, right) for line in open("spell-testset1.txt") for right, wrongs in [line.strip().split(":")] for wrong in wrongs.split()]
spell = SpellChecker()

def by_symspell(w):
    hits = sym.lookup(w, Verbosity.TOP, max_edit_distance=2)
    return hits[0].term if hits else w

def by_pyspellchecker(w):
    return spell.correction(w) or w

print(len(tests), "組 / 正解の語が symspellpy の辞書にない割合:", round(sum(r not in sym.words for _, r in tests) / len(tests), 3))
for name, f in [("symspellpy", by_symspell), ("pyspellchecker", by_pyspellchecker)]:
    t = time.perf_counter()
    out = [f(w) for w, _ in tests]
    sec = time.perf_counter() - t
    acc = sum(o == r for o, (_, r) in zip(out, tests)) / len(tests)
    print(f"{name}: 正解率 {acc:.3f} / {len(tests) / sec:.0f} 語/秒")
    print("   誤りの例 (入力, 出力, 正解):", [(w, o, r) for o, (w, r) in zip(out, tests) if o != r][:8])

270 組 / 正解の語が symspellpy の辞書にない割合: 0.015
symspellpy: 正解率 0.744 / 14905 語/秒
   誤りの例 (入力, 出力, 正解): [('contende', 'contend', 'contented'), ('contended', 'contended', 'contented'), ('proplen', 'people', 'problem'), ('exstacy', 'stacy', 'ecstasy'), ('guic', 'guin', 'juice'), ('juce', 'june', 'juice'), ('juse', 'use', 'juice'), ('localy', 'local', 'locally')]
pyspellchecker: 正解率 0.763 / 16 語/秒
   誤りの例 (入力, 出力, 正解): [('contende', 'contend', 'contented'), ('contended', 'contended', 'contented'), ('proplen', 'propmen', 'problem'), ('exstacy', 'eustacy', 'ecstasy'), ('guic', 'guib', 'juice'), ('juse', 'just', 'juice'), ('localy', 'local', 'locally'), ('compair', 'company', 'compare')]


In [4]:
# 文脈を使う: symspellpy の候補 (元の語も含む) を、マスク言語モデルで文脈に合う順に並べ替える
import re
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

torch.manual_seed(0)
name = "distilbert-base-uncased"     # Apache-2.0。小文字にしてから扱う
tokenizer = AutoTokenizer.from_pretrained(name)
mlm = AutoModelForMaskedLM.from_pretrained(name).eval()

def candidates(word, k=15):
    """編集距離 2 までの辞書の語のうち、頻度の高い k 語 (短い語は距離 1 まで)。辞書にある元の語は必ず残す"""
    hits = [h for h in sym.lookup(word, Verbosity.ALL, max_edit_distance=2) if h.distance <= 1 or len(word) >= 4]
    cands = {h.term: h.distance for h in sorted(hits, key=lambda h: -h.count)[:k]}
    if word in sym.words:
        cands[word] = 0
    return cands

@torch.no_grad()
def context_scores(words, i, cands):
    """i 番目の語を各候補に置き換えたときの、その語の対数確率 (複数トークンの語は、自分と後ろのトークンを隠して 1 トークンずつ予測して足す)"""
    left = tokenizer(" ".join(words[:i]), add_special_tokens=False)["input_ids"] if i else []
    right = tokenizer(" ".join(words[i + 1:]), add_special_tokens=False)["input_ids"]
    seqs, meta = [], []
    for c in cands:
        ids = tokenizer(c, add_special_tokens=False)["input_ids"]
        for j in range(len(ids)):
            span = ids[:j] + [tokenizer.mask_token_id] * (len(ids) - j)
            seqs.append([tokenizer.cls_token_id] + left + span + right + [tokenizer.sep_token_id])
            meta.append((c, 1 + len(left) + j, ids[j]))
    batch = tokenizer.pad({"input_ids": seqs}, return_tensors="pt")
    logp = mlm(**batch).logits.log_softmax(-1)
    scores = dict.fromkeys(cands, 0.0)
    for row, (c, pos, tok) in enumerate(meta):
        scores[c] += float(logp[row, pos, tok])
    return scores

def correct_sentence(sentence, penalty=2.0, margin=2.0):
    """左から 1 語ずつ直す。元の語から離れるほど減点し、辞書にある語は差が margin を超えたときだけ書き換える"""
    words = sentence.lower().split()
    for i, w in enumerate(words):
        if not re.fullmatch(r"[a-z]+", w):
            continue
        cands = candidates(w)
        if not cands or list(cands) == [w]:
            continue
        ctx = context_scores(words, i, cands)
        total = {c: ctx[c] - penalty * d for c, d in cands.items()}
        best = max(total, key=total.get)
        if best != w and (w not in cands or total[best] > total[w] + margin):
            words[i] = best
    return " ".join(words)

for s in ["I want a peace of cake .", "The whether is nice today .", "Their is a cat on the table .", "I like to read books ."]:
    print(s, "→", correct_sentence(s))

I want a peace of cake . → i want a piece of cake .
The whether is nice today . → the weather is nice today .
Their is a cat on the table . → there is a cat on the table .
I like to read books . → i like to read books .


In [5]:
# 手作りの文で比べる: 実在語の誤り (19 文)、辞書にない語の誤り (4 文)、誤りのない文 (12 文)
# Colab の CPU で 1〜3 分
data = """I want a peace of cake . || I want a piece of cake .
He will except the offer . || He will accept the offer .
She lives in a quite village . || She lives in a quiet village .
The whether is nice today . || The weather is nice today .
Their is a cat on the table . || There is a cat on the table .
I need to buy some bred . || I need to buy some bread .
Can you here me now ? || Can you hear me now ?
I lost my keys and could not find then . || I lost my keys and could not find them .
She is taller then her brother . || She is taller than her brother .
We swam in the see . || We swam in the sea .
Please turn of the light . || Please turn off the light .
He new the answer . || He knew the answer .
I have to brake the news to him . || I have to break the news to him .
It is to cold to swim . || It is too cold to swim .
I will meat you at the station . || I will meet you at the station .
She road her bike to school . || She rode her bike to school .
My farther is a doctor . || My father is a doctor .
We had a grate time at the party . || We had a great time at the party .
He has a dog and to cats . || He has a dog and two cats .
I recieved a wonderfull mesage from my freind . || I received a wonderful message from my friend .
The goverment anounced a new polcy . || The government announced a new policy .
She is a very inteligent studnet . || She is a very intelligent student .
I beleive this is the rigth answer . || I believe this is the right answer .
I like to read books . || I like to read books .
The cat sat on the mat . || The cat sat on the mat .
She bought a new car last week . || She bought a new car last week .
They went to the park after lunch . || They went to the park after lunch .
He plays the piano very well . || He plays the piano very well .
We are going to visit our grandparents . || We are going to visit our grandparents .
The weather was cold , so we stayed home . || The weather was cold , so we stayed home .
I think their idea is better than ours . || I think their idea is better than ours .
Can you hear the birds singing ? || Can you hear the birds singing ?
Please put the box over there . || Please put the box over there .
He accepted the job offer . || He accepted the job offer .
The quiet library is near the station . || The quiet library is near the station ."""
pairs = [line.split(" || ") for line in data.split("\n")]
kinds = ["実在語の誤り"] * 19 + ["辞書にない語の誤り"] * 4 + ["誤りなし"] * 12   # data の文の順
known = set(sym.words)

def words_of(s):
    """記号を除いた小文字の語の列 (lookup_compound は記号を落とすので、記号を除いて比べる)"""
    return re.findall(r"[a-z]+", s.lower())

def is_word(w):
    return re.fullmatch(r"[a-z]+", w) is not None

def word_by_word_symspell(s):
    """文脈を見ずに、symspellpy の辞書にない語だけを 1 語ずつ直す"""
    return " ".join(by_symspell(w) if is_word(w) and w not in known else w for w in s.lower().split())

def word_by_word_pyspellchecker(s):
    """文脈を見ずに、pyspellchecker の辞書にない語だけを 1 語ずつ直す"""
    return " ".join(by_pyspellchecker(w) if is_word(w) and spell.unknown([w]) else w for w in s.lower().split())

def compound(s):
    return sym.lookup_compound(" ".join(words_of(s)), max_edit_distance=2)[0].term

systems = {"pyspellchecker (1 語ずつ)": word_by_word_pyspellchecker,
           "symspellpy (1 語ずつ)": word_by_word_symspell,
           "symspellpy の lookup_compound": compound,
           "symspellpy + マスク言語モデル": correct_sentence}

for sys_name, f in systems.items():
    t = time.perf_counter()
    n_err = {"実在語の誤り": 0, "辞書にない語の誤り": 0}
    fixed = {"実在語の誤り": 0, "辞書にない語の誤り": 0}
    miscorrected = wrong = 0
    failures = []
    for (src, ref), kind in zip(pairs, kinds):
        s, r, out = words_of(src), words_of(ref), words_of(f(src))
        if len(out) != len(s):
            # 語の数が変わった (分割・結合した) 文は、誤りを直せず、正しい語を 1 語書き換えたと数える
            if kind in n_err:
                n_err[kind] += sum(a != b for a, b in zip(s, r))
            wrong += 1
            failures.append(" ".join(out))
            continue
        for a, o, b in zip(s, out, r):
            if a != b:
                n_err[kind] += 1
                fixed[kind] += (o == b)
                miscorrected += (o != b and o != a)
            elif o != b:
                wrong += 1                      # 正しい語を書き換えた
        if out != r:
            failures.append(" ".join(out))
    sec = time.perf_counter() - t
    detail = " / ".join(f"{k} {fixed[k]}/{n_err[k]}" for k in n_err)
    print(f"{sys_name}: 直せた {sum(fixed.values())}/{sum(n_err.values())} 語 ({detail}) / 別の誤った語にした {miscorrected} / 正しい語を書き換えた {wrong} / {sec:.1f} 秒")
    print("   直しきれなかった文の例:", failures[:4])

pyspellchecker (1 語ずつ): 直せた 11/30 語 (実在語の誤り 0/19 / 辞書にない語の誤り 11/11) / 別の誤った語にした 0 / 正しい語を書き換えた 0 / 0.0 秒
   直しきれなかった文の例: ['i want a peace of cake', 'he will except the offer', 'she lives in a quite village', 'the whether is nice today']
symspellpy (1 語ずつ): 直せた 11/30 語 (実在語の誤り 0/19 / 辞書にない語の誤り 11/11) / 別の誤った語にした 0 / 正しい語を書き換えた 0 / 0.0 秒
   直しきれなかった文の例: ['i want a peace of cake', 'he will except the offer', 'she lives in a quite village', 'the whether is nice today']
symspellpy の lookup_compound: 直せた 11/30 語 (実在語の誤り 0/19 / 辞書にない語の誤り 11/11) / 別の誤った語にした 0 / 正しい語を書き換えた 0 / 0.1 秒
   直しきれなかった文の例: ['i want a peace of cake', 'he will except the offer', 'she lives in a quite village', 'the whether is nice today']
symspellpy + マスク言語モデル: 直せた 26/30 語 (実在語の誤り 15/19 / 辞書にない語の誤り 11/11) / 別の誤った語にした 4 / 正しい語を書き換えた 0 / 38.0 秒
   直しきれなかった文の例: ['i need to buy some red', 'can you help me now', 'i will get you at the station', 'she had her bike to school']


- この章の「英語のスペルミスをライブラリで直す (pyspellchecker)」は、編集距離と頻度で 1 語ずつ直す方法で、文脈を見ない。このレシピは候補を出す部分を symspellpy にし、選ぶ部分にマスク言語モデルを使って、実在する別の語になってしまった誤り（real-word error。peace → piece、whether → weather）も直す。
- symspellpy は Wolf Garbe の SymSpell (C#) を Python に移したもので、MIT ライセンス。辞書の語から文字を削除した形をあらかじめ索引にしておき、入力からも削除した形を作って引くので、候補を速く引ける。同梱の辞書は SymSpell の README によると Google Books Ngram (CC BY 3.0) と SCOWL から作られている。`lookup_compound` は語の分割・結合の誤り（whereis、th elove）もまとめて直せる。
- Norvig のテストセット（270 組、1 語ずつ）では、正解率は symspellpy が 0.744、pyspellchecker が 0.763 で、差は 5 語ほどなのでほぼ同じと見てよい。速さは大きく違い、この実行では symspellpy が毎秒約 1.5 万語、pyspellchecker が毎秒 16 語だった（pyspellchecker は編集距離 2 の文字列をすべて作って辞書と照合する）。大量の文書を処理するなら symspellpy がよい。誤りの例を見ると、失敗の傾向も 2 つで似ている。正解より近い語があればそちらを選び（contende → contend、symspellpy の juse → use）、同じ距離なら頻度の高い語を選ぶ（localy → local、symspellpy の proplen → people）。入力が実在語 (contended → contented) だと直せない。
- 手作りの 35 文では、文脈を見ない 3 つの方法（lookup_compound も含む）はどれも、辞書にない語の誤り 11 語はすべて直せたが、実在語の誤り 19 語は 1 つも直せなかった。マスク言語モデルで選ぶと 30 語中 26 語（実在語の誤り 15/19、辞書にない語の誤り 11/11）を直し、誤りのない 12 文は 1 語も書き換えなかった。ただし文の数が少なく、減点 (`penalty`) としきい値 (`margin`) はこの 35 文を見ながら決めた値なので、自分のデータで確かめてから使う。
- 直せなかった 4 語（bred → red、here → help、meat → get、road → had）は、どれも文としては自然だが意味の違う語に書き換えている。マスク言語モデルは「その位置に来やすい語」を選ぶので、よく使われる語に引き寄せられる。`margin` を大きくすると書き換えが減り、見逃しが増える。教育の場面では、自動で書き換えずに候補として示すほうが安全である。
- 文脈を使う訂正は遅い。この実行では 2 コアの CPU で 35 文に 38 秒（1 文 1 秒ほど）かかり、symspellpy だけなら 0.1 秒以下だった。大量に処理するなら GPU を使う、取り違えやすい語の組（their / there、then / than など）だけを調べる、といった工夫がいる。
- 小文字にしてから扱うので、大文字・小文字の情報は失われる。辞書にない語は、編集距離 2 以内に候補があれば必ずどれかに書き換えるので、固有名詞や専門用語は `sym.create_dictionary_entry("tokenizer", 1000)` のように辞書に足しておく。学習者のつづりの誤りは打ち間違いと傾向が違う（発音に引きずられる、母語の影響を受ける）ので、学習者の作文で確かめる。日本語では、スペル誤りにあたるのは誤変換や脱字である。日本語入力の章と、この章の「マスク言語モデルで誤りのありそうな箇所を見つける（文法誤り検出）」を参照。
- 仕組みを自分で実装して確かめる版（編集距離 1・2 の候補を作り、頻度で選ぶ Norvig 方式）は、付録「仕組みを理解するための実装」の「編集距離と単語の頻度で英語のスペル誤りを訂正する（Norvig 方式を自分で実装）」にある。